# 01 · 38-Cloud data exploration

Goal: verify, with the **real dataset**, every assumption the pipeline makes before any training happens.
Nothing in this notebook is simulated. Cells that need data stop with a clear error if it is missing.

Checks performed
1. Folder layout, CSV files and the number of patches/scenes against the official counts.
2. Band matching **by patch name**, and file dtypes/shapes.
3. Ground-truth label encoding, and whether any cloud labels fall on nodata pixels.
4. Share of mostly-empty patches vs the official `*_nonempty.csv`.
5. Test-scene padding geometry against the reference masks in `Entire_scene_gts`.
6. Class balance, sample visualization, and the deterministic scene-level split.

**Before running**: set `STRATOSENSE_DATA_ROOT` to the folder containing the extracted `38-Cloud_training` and `38-Cloud_test`
directories (Kaggle: `sorour/38cloud-cloud-segmentation-in-satellite-images`; see `docs/dataset.md`).

## 0 · Setup

In [ ]:
# On Google Colab, uncomment and edit:
# !git clone <YOUR_REPO_URL> StratoSense && cd StratoSense
# %cd StratoSense
# !pip install -q -r ml/requirements.txt
import os, sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "ml").is_dir():          # notebook lives in notebooks/
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from ml.config import (BAND_ORDER, OFFICIAL_TEST_PATCHES, OFFICIAL_TEST_SCENES, OFFICIAL_TRAIN_PATCHES,
                       PATCH_SIZE, SplitConfig, data_root_from_env)
from ml.dataset import (DatasetError, DatasetLayout, build_patch_index, filter_nonempty, load_patch,
                        read_names_csv, records_for_scenes, split_scenes, unique_scene_ids)
from ml.explore import layout_summary, sample_records, scan, scene_table, summarize_scan
from ml.visualize import rgb_composite
from ml.preprocessing import compute_valid_mask

DATA_ROOT = data_root_from_env()
SCAN_LIMIT = int(os.environ.get("STRATOSENSE_SCAN_LIMIT", "0")) or None   # None = scan every patch
print("Data root:", DATA_ROOT)

## 1 · Layout, CSVs and official counts

In [ ]:
train_layout = DatasetLayout.discover(DATA_ROOT, "train")
train_records = build_patch_index(train_layout, with_gt=True)   # raises if any band/GT file is missing
print(pd.Series(layout_summary(train_layout, train_records)).to_string())
print(f"\nOfficial training patches: {OFFICIAL_TRAIN_PATCHES} | found: {len(train_records)} "
      f"-> {'MATCH' if len(train_records) == OFFICIAL_TRAIN_PATCHES else 'MISMATCH, investigate'}")

try:
    test_layout = DatasetLayout.discover(DATA_ROOT, "test")
    test_records = build_patch_index(test_layout, with_gt=False)
    print(f"Official test patches: {OFFICIAL_TEST_PATCHES} | found: {len(test_records)} "
          f"-> {'MATCH' if len(test_records) == OFFICIAL_TEST_PATCHES else 'MISMATCH, investigate'}")
    print(f"Official test scenes: {OFFICIAL_TEST_SCENES} | found: {len(unique_scene_ids(test_records))}")
except DatasetError as exc:
    test_layout = test_records = None
    print("Test split not available:", exc)

## 2 · Band matching and a patch-level sanity check
`build_patch_index` matched every patch to its four band files **by name** (never by directory order).
Here we load one patch and confirm dtype, shape and value ranges for each band in model order.

In [ ]:
rec = train_records[0]
raw, mask = load_patch(rec)
print(rec.patch.name)
print("raw:", raw.shape, raw.dtype, "| mask:", mask.shape, mask.dtype)
for i, b in enumerate(BAND_ORDER):
    print(f"  {b:5s} <- {rec.band_paths[i].name:80s} min={raw[i].min():6d} max={raw[i].max():6d}")

## 3 · Train scene inventory
Each scene is cut into a grid of 384×384 tiles. A complete grid is required for the test-scene reconstruction.

In [ ]:
train_scenes_df = pd.DataFrame(scene_table(train_records))
display(train_scenes_df)
print("All training grids complete:", bool(train_scenes_df.grid_complete.all()))
if test_records:
    test_scenes_df = pd.DataFrame(scene_table(test_records))
    display(test_scenes_df)
    print("All test grids complete:", bool(test_scenes_df.grid_complete.all()))

## 4 · Test-scene padding geometry (verifies the reconstruction assumption)
The official evaluation stitches tiles into a padded scene and removes the padding with a **centred** crop to the
reference mask size. Here we read the real reference masks and check that every reconstructed grid is at least as large
as its mask and less than one tile larger on each axis. If this fails, `ml/scene.py` must be revisited before evaluation.

In [ ]:
import rasterio
if test_records and test_layout.scene_gt_dir is not None:
    rows = []
    for _, r in test_scenes_df.iterrows():
        candidates = list(test_layout.scene_gt_dir.glob(f"*{r.scene_id}*"))
        if len(candidates) != 1:
            rows.append({"scene_id": r.scene_id, "gt_file": None, "ok": False}); continue
        with rasterio.open(candidates[0]) as src:
            gh, gw, dtype = src.height, src.width, src.dtypes[0]
            vals = np.unique(src.read(1))
        dy, dx = r.padded_height - gh, r.padded_width - gw
        rows.append({"scene_id": r.scene_id, "gt_file": candidates[0].name, "gt_h": gh, "gt_w": gw,
                     "pad_h": dy, "pad_w": dx, "gt_dtype": dtype, "gt_values": vals.tolist(),
                     "ok": 0 <= dy < PATCH_SIZE and 0 <= dx < PATCH_SIZE})
    geometry = pd.DataFrame(rows); display(geometry)
    print("Padding assumption holds for all scenes:", bool(geometry.ok.all()))
else:
    print("Skipped: test split or Entire_scene_gts not available.")

## 5 · Full scan of the training patches
Valid-pixel fraction (non-zero margin excluded), label values and cloud share per patch.
`decode_mask` raises if any label is not 0/255, so reaching the summary means the encoding is as documented.
Set `STRATOSENSE_SCAN_LIMIT` to scan a subset while testing.

In [ ]:
subset = train_records if SCAN_LIMIT is None else sample_records(train_records, SCAN_LIMIT, seed=0)
scan_df = pd.DataFrame(scan(subset, progress=lambda xs: tqdm(xs)))
summary = summarize_scan(scan_df.to_dict("records"))
print(pd.Series(summary).to_string())

### Mostly-empty patches vs the official `nonempty` list
The dataset README describes a list of informative patches (>80% non-zero pixels). We compare it with our own count.

In [ ]:
if train_layout.nonempty_csv is None:
    print("No *_nonempty.csv shipped with this copy of the dataset; use the >=80% valid-fraction rule from the scan.")
else:
    official = set(read_names_csv(train_layout.nonempty_csv))
    ours = set(scan_df.loc[scan_df.valid_fraction >= 0.8, "name"])
    scanned = set(scan_df.name)
    print(f"official nonempty: {len(official)} | scanned >=80% valid: {len(ours)}")
    print("scanned patches only in official list:", len((official & scanned) - ours))
    print("scanned patches only in our >=80% set:", len(ours - official))

## 6 · Class balance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(scan_df.valid_fraction, bins=30); ax[0].set_title("Valid-pixel fraction per patch")
ax[1].hist(scan_df.cloud_fraction_of_valid.dropna(), bins=30); ax[1].set_title("Cloud fraction of valid pixels")
plt.tight_layout(); plt.show()
print("Mean cloud fraction over valid pixels:", round(scan_df.cloud_fraction_of_valid.mean(), 4))

## 7 · Samples (RGB composite, NIR, ground truth)

In [ ]:
picks = sample_records([r for r in train_records], 4, seed=3)
fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
for row, rec in zip(np.atleast_2d(axes), picks):
    raw, mask = load_patch(rec); valid = compute_valid_mask(raw)
    row[0].imshow(rgb_composite(raw, valid)); row[0].set_title("RGB (display stretch)")
    row[1].imshow(raw[BAND_ORDER.index("nir")], cmap="gray"); row[1].set_title("NIR")
    row[2].imshow(mask, cmap="gray", vmin=0, vmax=1); row[2].set_title("Ground truth")
    for a in row: a.axis("off")
plt.tight_layout(); plt.show()

## 8 · Deterministic scene-level split
Validation scenes are held out **whole**, so patches from one geographic scene never sit on both sides.
(The dataset authors' own example code splits randomly by patch, which leaks neighbouring tiles.)

In [ ]:
base = [r for r in train_records]
train_scenes, val_scenes = split_scenes(unique_scene_ids(base), SplitConfig())
n_tr, n_va = len(records_for_scenes(base, train_scenes)), len(records_for_scenes(base, val_scenes))
print(f"train scenes: {len(train_scenes)} ({n_tr} patches) | validation scenes: {len(val_scenes)} ({n_va} patches)")
print("validation scenes:", val_scenes)
assert not set(train_scenes) & set(val_scenes)

## 9 · Next step: normalization statistics
Computed from the **training scenes only** and saved to `artifacts/normalization.json`:

```
python -m ml.compute_stats --data-root $STRATOSENSE_DATA_ROOT
```

Record the findings from this notebook (patch counts, padding check, nonempty comparison) in `docs/dataset.md`
before moving on to model training.